# Input preparation: metadata, frames, transcripts (300 ads)
Run once (GPU T4, Internet on), then **Save Version → Save & Run All**. The output folder
(`ad_meta_300.csv`, `frames/`, `frames_index.csv`, `transcripts_300.csv`, `ad_inputs_300.jsonl`)
becomes the input for every model run (OpenAI now, Gemini later), so all models see identical information.

Inputs to attach: the ads dataset, and the experiment notebook's output (for `ad_sample_300.csv`, `guidelines.txt`).

In [ ]:
# P0. Setup
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "faster-whisper==1.2.1", "imageio-ffmpeg==0.6.0"], check=True)

import ast, glob, json, math, os, re, shutil
from pathlib import Path
os.environ["OPENCV_FFMPEG_LOGLEVEL"] = "-8"          # silence decoder warnings
import numpy as np, pandas as pd, cv2, imageio_ffmpeg

DATA = Path("/kaggle/input/<dataset>/uploaded_data/uploaded_data")
WORK = Path("/kaggle/working")
FRAMES = WORK / "frames"; FRAMES.mkdir(parents=True, exist_ok=True)

INCLUDE_ADVERTISER_LOCATION = False   # can hint at a culture; off by default
N_FRAMES_MAX = 16                     # frames per ad, evenly spaced over the whole ad
FRAME_LONG_SIDE = 512                 # px; keeps image tokens and upload size small
WHISPER_MODEL = "large-v3"

def find_input(name):
    hits = sorted(glob.glob(f"/kaggle/input/notebooks/*/*/{name}")) or sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found: attach the experiment notebook's output as an input"
    return Path(hits[0])

SAMPLE_CSV, GUIDE = find_input("ad_sample_300.csv"), find_input("guidelines.txt")
shutil.copy(SAMPLE_CSV, WORK / "ad_sample_300.csv"); shutil.copy(GUIDE, WORK / "guidelines.txt")
sample = pd.read_csv(WORK / "ad_sample_300.csv")
sample["ad_id_clean"] = sample["ad_id_clean"].astype(str)

STATIC_FFMPEG = str(next((Path(imageio_ffmpeg.__file__).parent / "binaries").glob("ffmpeg-*")))
Path(STATIC_FFMPEG).chmod(0o755)
print("sample:", SAMPLE_CSV, "| ads:", len(sample), "| classes:", sample.primary_tag.value_counts().to_dict())
print("guidelines:", GUIDE)
print(subprocess.run([STATIC_FFMPEG, "-version"], capture_output=True, text=True).stdout.splitlines()[0])

## P1. Metadata
From `ad_metadata.csv` (title, description, channel, category, language, tags, duration) and
`videos.parquet` (advertiser). **Never included:** `primary_tag` / `secondary_tag` (the human labels),
`country` (where the ad was shown), engagement counts.

In [ ]:
def clean(x, cap=None):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ""
    s = re.sub(r"\s+", " ", str(x)).strip()
    if s.lower() in {"nan", "none", "null", "[]"}:
        return ""
    if cap and len(s) > cap:
        s = s[:cap].rsplit(" ", 1)[0] + " [...]"
    return s

def clean_tags(x, cap=300):
    s = clean(x)
    items = [(a or b).strip() for a, b in re.findall(r"'([^'\[\]]+)'|\"([^\"\[\]]+)\"", s)]
    items = [i for i in items if i]
    if items:
        s = ", ".join(items)
    elif not s.strip("[]'\" ,"):          # e.g. "['']" or "[]"
        s = ""
    return clean(s, cap)

meta = pd.read_csv(DATA / "ad_metadata.csv", dtype={"Ad ID": str})
meta["Ad ID"] = meta["Ad ID"].str.strip()
print(f"ad_metadata.csv: {len(meta)} rows, {meta['Ad ID'].nunique()} unique Ad IDs")
meta = (meta.assign(_filled=meta.notna().sum(axis=1)).sort_values("_filled", ascending=False)
            .drop_duplicates("Ad ID").drop(columns="_filled"))

vids = (pd.read_parquet(DATA / "videos.parquet", columns=["video_id", "advertiser_name", "advertiser_location"])
          .assign(video_id=lambda d: d.video_id.astype(str)).drop_duplicates("video_id"))

M = (sample[["ad_id_clean"]]
     .merge(meta, left_on="ad_id_clean", right_on="Ad ID", how="left")
     .merge(vids, left_on="ad_id_clean", right_on="video_id", how="left"))

META = pd.DataFrame({
    "ad_id": M.ad_id_clean,
    "has_metadata": M["Ad ID"].notna(),
    "title": M["Ad Title"].map(lambda x: clean(x, 200)),
    "description": M["Description"].map(lambda x: clean(x, 1000)),
    "channel": M["Channel Title"].map(clean),
    "category": M["Category Name"].map(clean),
    "language": M["Language"].map(clean),
    "tags": M["Tags"].map(clean_tags),
    "duration_s": M["Duration (seconds)"],
    "advertiser": M["advertiser_name"].map(clean),
    "advertiser_location": M["advertiser_location"].map(clean) if INCLUDE_ADVERTISER_LOCATION else "",
})
assert not {"primary_tag", "secondary_tag", "label"} & set(META.columns)
META.to_csv(WORK / "ad_meta_300.csv", index=False)

print(f"\nmatched in ad_metadata.csv: {META.has_metadata.sum()} / {len(META)}")
print("non-empty share per field:")
print((META.drop(columns=["ad_id", "has_metadata", "duration_s"]) != "").mean().round(2).to_string())
display(META.head(3).T)

## P2. Frames
Up to 16 frames per ad, evenly spaced over the **whole** ad (one per second for ads shorter than 16 s),
resized to 512 px on the long side. Decoded with OpenCV; if a frame fails, the pinned static ffmpeg
is used (libaom decoder for AV1, which avoids the crash seen earlier).

In [ ]:
def probe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type,codec_name",
                        "-of", "json", str(path)], capture_output=True, text=True)
    j = json.loads(r.stdout or "{}")
    streams = j.get("streams", [])
    vcodec = next((s["codec_name"] for s in streams if s.get("codec_type") == "video"), None)
    has_audio = any(s.get("codec_type") == "audio" for s in streams)
    return float(j.get("format", {}).get("duration", "nan")), vcodec, has_audio

def frame_times(dur):
    n = int(min(N_FRAMES_MAX, max(1, math.ceil(dur))))
    return [round((i + 0.5) * dur / n, 2) for i in range(n)]

def resize(img):
    h, w = img.shape[:2]
    s = FRAME_LONG_SIDE / max(h, w)
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA) if s < 1 else img

def grab_ffmpeg(src, t, dst, vcodec):
    dec = ["-c:v", "libaom-av1"] if vcodec == "av1" else []
    subprocess.run([STATIC_FFMPEG, "-y", "-v", "error", *dec, "-ss", f"{t:.2f}", "-i", str(src),
                    "-frames:v", "1", "-vf", f"scale='if(gt(iw,ih),{FRAME_LONG_SIDE},-2)':'if(gt(iw,ih),-2,{FRAME_LONG_SIDE})'",
                    "-q:v", "3", str(dst)], capture_output=True)
    return dst.exists() and dst.stat().st_size > 0

rows, fails = [], []
probes = {}
for i, ad in enumerate(sample.ad_id_clean, 1):
    src = DATA / "videos" / f"{ad}.mp4"
    dur, vcodec, has_audio = probe(src)
    probes[ad] = {"duration": dur, "vcodec": vcodec, "has_audio": has_audio}
    cap = cv2.VideoCapture(str(src))
    for k, t in enumerate(frame_times(dur)):
        dst = FRAMES / f"{ad}_f{k:02d}.jpg"
        method = "cached"
        if not dst.exists():
            ok, img = False, None
            if cap.isOpened():
                cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
                ok, img = cap.read()
            if ok and img is not None:
                cv2.imwrite(str(dst), resize(img), [cv2.IMWRITE_JPEG_QUALITY, 85]); method = "opencv"
            elif grab_ffmpeg(src, t, dst, vcodec):
                method = "ffmpeg"
            else:
                fails.append((ad, t)); continue
        rows.append({"ad_id": ad, "k": k, "t": t, "file": dst.name, "method": method})
    cap.release()
    if i % 25 == 0:
        print(f"{i}/{len(sample)} ads")

FIDX = pd.DataFrame(rows)
FIDX.to_csv(WORK / "frames_index.csv", index=False)
pd.DataFrame.from_dict(probes, orient="index").rename_axis("ad_id").to_csv(WORK / "probe_300.csv")
print(f"\nframes: {len(FIDX)} for {FIDX.ad_id.nunique()} ads | failed frames: {len(fails)}")
print("frames per ad:", FIDX.groupby("ad_id").size().describe()[["min", "50%", "max"]].to_dict())
print("method:", FIDX.method.value_counts().to_dict())
print(f"frames folder: {sum(p.stat().st_size for p in FRAMES.iterdir())/1e6:.1f} MB")

## P3. Audio transcripts (GPU)
Whisper large-v3 over each ad's **full** audio (audio is decoded alone, so the video crash can't occur).
Voice-activity detection removes music and silence (Whisper otherwise invents text there); temperature 0
and beam 5 keep it reproducible. Non-English speech is also translated to English.
Resumable: re-running skips ads already in `transcripts_300.csv`.

In [ ]:
import ctypes, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":   # make sure ctranslate2 finds the cuDNN/cuBLAS that ship with torch
    for mod in ("nvidia.cudnn", "nvidia.cublas"):
        try:
            libdir = Path(__import__(mod, fromlist=["_"]).__path__[0]) / "lib"
            for lib in sorted(libdir.glob("lib*.so*")):
                ctypes.CDLL(str(lib), mode=ctypes.RTLD_GLOBAL)
        except Exception as e:
            print("note:", mod, e)
from faster_whisper import WhisperModel
model = WhisperModel(WHISPER_MODEL, device=DEVICE, compute_type="float16" if DEVICE == "cuda" else "int8")
print("Whisper", WHISPER_MODEL, "on", DEVICE)

def run_whisper(audio, task, language=None):
    segs, info = model.transcribe(audio, task=task, language=language, beam_size=5, temperature=0.0,
                                  vad_filter=True, condition_on_previous_text=False)
    segs = list(segs)
    return " ".join(s.text.strip() for s in segs).strip(), info, segs

OUT = WORK / "transcripts_300.csv"
done = set(pd.read_csv(OUT, dtype={"ad_id": str}).ad_id) if OUT.exists() else set()
for i, ad in enumerate(sample.ad_id_clean, 1):
    if ad in done:
        continue
    src = DATA / "videos" / f"{ad}.mp4"
    rec = {"ad_id": ad, "has_audio": probes[ad]["has_audio"], "language": "", "language_prob": np.nan,
           "speech_seconds": 0.0, "text_original": "", "text_en": "", "n_words": 0, "mean_logprob": np.nan,
           "status": "no_audio"}
    if probes[ad]["has_audio"]:
        # decode audio only, straight to 16 kHz mono float32 (bypasses faster-whisper's own file reader)
        r = subprocess.run([STATIC_FFMPEG, "-v", "error", "-i", str(src), "-vn", "-ac", "1", "-ar", "16000",
                            "-f", "f32le", "-acodec", "pcm_f32le", "-"], capture_output=True)
        audio = np.frombuffer(r.stdout, dtype=np.float32)
        if r.returncode != 0 or audio.size == 0:
            rec["status"] = f"audio_error: {r.stderr.decode(errors='ignore').strip()[-150:]}"
        else:
            text, info, segs = run_whisper(audio, "transcribe")
            rec.update(language=info.language, language_prob=round(info.language_probability, 3),
                       speech_seconds=round(sum(s.end - s.start for s in segs), 1), text_original=text,
                       n_words=len(text.split()),
                       mean_logprob=round(float(np.mean([s.avg_logprob for s in segs])), 3) if segs else np.nan)
            if not text:
                rec["status"] = "no_speech"
            else:
                rec["status"] = "ok"
                rec["text_en"] = text if info.language == "en" else run_whisper(audio, "translate", info.language)[0]
    pd.DataFrame([rec]).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    if i % 10 == 0:
        print(f"{i}/{len(sample)} ads")

TR = pd.read_csv(OUT, dtype={"ad_id": str}, keep_default_na=False)
print("\nstatus:", TR.status.value_counts().to_dict())
print("top languages:", TR.loc[TR.status == "ok", "language"].value_counts().head(10).to_dict())
print("words per ad (with speech):", pd.to_numeric(TR.loc[TR.status == "ok", "n_words"]).describe()[["50%", "max"]].to_dict())

## P4. Combine and preview what the model will read
One record per ad: metadata + transcript + frame list. The run notebook builds prompts only from
`ad_inputs_300.jsonl`. Check the preview below for the kinds of problems Whisper can introduce
(text invented over music, wrong language).

In [ ]:
TRANSCRIPT_MAX_WORDS = 600

def mmss(s):
    s = float(s); return f"{int(s // 60)}:{int(round(s % 60)):02d}"

def context_text(m, tr, dur):
    lines = ["Advertisement metadata:"]
    for key, label in [("title", "Title"), ("advertiser", "Advertiser"), ("channel", "YouTube channel"),
                       ("category", "YouTube category"), ("language", "Language"), ("tags", "Tags"),
                       ("description", "Description"), ("advertiser_location", "Advertiser location")]:
        if m.get(key):
            lines.append(f"- {label}: {m[key]}")
    lines.append(f"- Duration: {mmss(dur)}")
    lines.append("")
    if tr["status"] == "ok":
        words = tr["text_en"].split()
        txt = " ".join(words[:TRANSCRIPT_MAX_WORDS]) + (" [...]" if len(words) > TRANSCRIPT_MAX_WORDS else "")
        lang = "" if tr["language"] == "en" else f" (spoken language: {tr['language']}; English translation shown)"
        lines.append(f"Audio transcript, generated automatically and may contain errors{lang}:\n\"{txt}\"")
    elif tr["status"] == "no_speech":
        lines.append("Audio transcript: no speech detected (music or sound only).")
    else:
        lines.append("Audio transcript: the ad has no audio track.")
    return "\n".join(lines)

META = pd.read_csv(WORK / "ad_meta_300.csv", dtype=str, keep_default_na=False)
TR = pd.read_csv(WORK / "transcripts_300.csv", dtype=str, keep_default_na=False)
FIDX = pd.read_csv(WORK / "frames_index.csv", dtype={"ad_id": str})
PROBE = pd.read_csv(WORK / "probe_300.csv", dtype={"ad_id": str}).set_index("ad_id")

recs = []
for ad in sample.ad_id_clean:
    m = META.set_index("ad_id").loc[ad].to_dict()
    tr = TR.set_index("ad_id").loc[ad].to_dict()
    f = FIDX[FIDX.ad_id == ad].sort_values("k")
    recs.append({"ad_id": ad, "duration": PROBE.at[ad, "duration"],
                 "context": context_text(m, tr, PROBE.at[ad, "duration"]),
                 "frames": [{"file": r.file, "t": r.t, "label": f"Frame at {mmss(r.t)}"} for r in f.itertuples()],
                 "transcript_status": tr["status"], "transcript_language": tr["language"]})
with open(WORK / "ad_inputs_300.jsonl", "w") as fh:
    for r in recs:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"ad_inputs_300.jsonl: {len(recs)} ads | frames per ad: "
      f"{min(len(r['frames']) for r in recs)}-{max(len(r['frames']) for r in recs)}")
lens = [len(r["context"].split()) for r in recs]
print(f"context length (words): median {int(np.median(lens))}, max {max(lens)}")
for r in recs[:2]:
    print("\n" + "=" * 100 + f"\n{r['ad_id']}  ({len(r['frames'])} frames)\n" + r["context"])